In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:

df = pd.read_csv('../data/universal_training_data.csv')

features = ['Open', 'High', 'Low', 'Close', 'Volume','SMA_20_pct', 'SMA_50_pct', 'BB_Upper_pct','BB_Lower_pct', 'RSI', 'Daily_Return']
target = 'Next_Day_Return'

train_df = df[df['Dataset_Tag'] == 'Train']
test_df  = df[df['Dataset_Tag'] == 'Test']

X_train, y_train = train_df[features], train_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

print(f"Training on {len(X_train):,} days")
print(f"Testing on  {len(X_test):,} days\n")

In [ ]:

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred  = lr_model.predict(X_test)

rf_model = RandomForestRegressor( n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

gbr_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
gbr_model.fit(X_train, y_train)
gbr_pred = gbr_model.predict(X_test)

In [ ]:

def evaluate(name, preds, actual):
    rmse    = np.sqrt(mean_squared_error(actual, preds))
    mae     = mean_absolute_error(actual, preds)
    r2      = r2_score(actual, preds)
    dir_acc = np.mean((preds >= 0) == (actual.values >= 0)) * 100
    print(f"{name}")
    print(f"  RMSE:                 {rmse:.5f}")
    print(f"  MAE:                  {mae:.5f}")
    print(f"  R² Score:             {r2:.4f}")
    print(f"  Directional Accuracy: {dir_acc:.1f}%\n")

evaluate('Linear Regression', lr_pred, y_test)
evaluate('Random Forest',     rf_pred, y_test)
evaluate('Gradient Boosting', gbr_pred, y_test)

In [ ]:
importance = pd.DataFrame({'Feature': features, 'Importance': rf_model.feature_importances_})
importance = importance.sort_values( by='Importance', ascending=False)

print("Feature Importance:\n")
print(importance)

In [ ]:
results = test_df[['Name']].copy()
results['Actual_Return'] = y_test.values
results['Predicted_Return'] = rf_pred
results.to_csv( '../data/predicted_portfolio_signals.csv',index=False)